In [5]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import datetime
import sys
import os
import sqlite3
import requests
import geopandas as gpd
import time

from pathlib import Path

from python.db import init_db, get_connection
from python.users import create_user, get_user, list_users, update_user


In [6]:
init_db() 

Database ready: data\jobs.db


In [7]:
# Pure function test
create_user(
    name="Star POO",
    city="Austin",
    state="TX",
    lat=30.27,
    lon=-97.74,
    preferred_pay_min=110000,
    preferred_pay_max=160000,
    max_weekly_commute_miles=80,
    notes="Primary profile"
)

print(get_user(1))
print(list_users())

# Update test
update_user(1, preferred_pay_min=120000, notes="Updated min pay")
print(get_user(1))


{'user_id': 1, 'name': 'Star POO', 'city': 'Austin', 'state': 'TX', 'lat': 30.27, 'lon': -97.74, 'preferred_pay_min': 110000.0, 'preferred_pay_max': 160000.0, 'max_weekly_commute_miles': 80.0, 'notes': 'Primary profile'}
[{'user_id': 1, 'name': 'Star POO', 'city': 'Austin', 'state': 'TX', 'lat': 30.27, 'lon': -97.74, 'preferred_pay_min': 110000.0, 'preferred_pay_max': 160000.0, 'max_weekly_commute_miles': 80.0, 'notes': 'Primary profile'}]
{'user_id': 1, 'name': 'Star POO', 'city': 'Austin', 'state': 'TX', 'lat': 30.27, 'lon': -97.74, 'preferred_pay_min': 120000.0, 'preferred_pay_max': 160000.0, 'max_weekly_commute_miles': 80.0, 'notes': 'Updated min pay'}


In [8]:
"""with get_connection() as conn:
    conn.execute("DELETE FROM user WHERE user_id = 1")
    conn.commit()
    print("User 1 deleted.")"""

'with get_connection() as conn:\n    conn.execute("DELETE FROM user WHERE user_id = 1")\n    conn.commit()\n    print("User 1 deleted.")'

```markdown
# Star Hound Tracker – Development Notebook Reference

**Personal job-search tracker**  
Local-first · SQLite is the source of truth · pandas + matplotlib/seaborn for analysis & plots

---

## 1. High-level Architecture

```text
User input / menu
  → SQL INSERT / UPDATE / SELECT on SQLite (data/jobs.db)
    → pandas (pd.read_sql) when analysis is needed
      → matplotlib / seaborn → plots/ and reports/
```

- **Writes & pipeline updates** → SQLite  
- **Visuals & reports** → query → DataFrame → plot  
- Always run: `PRAGMA foreign_keys = ON` on every connection

---

## 2. High-level Flow (V1)

1. Maintain a simple **user** profile (home location + pay preferences)
2. Add **jobs** manually
3. Compute a **job score** from pay + weekly commute + manual fit
4. Move promising jobs into **applications** and track status over time
5. Surface follow-ups and basic charts

---

## 3. Data Model (V1)

**Database file:** `data/jobs.db`

### `user` table (single row for V1)
| Column                     | Type    | Notes                              |
|----------------------------|---------|------------------------------------|
| user_id                    | INTEGER | PRIMARY KEY (usually 1)            |
| name                       | TEXT    | Optional but useful                |
| city                       | TEXT    | Home city                          |
| state                      | TEXT    | Home state                         |
| lat                        | REAL    | Home latitude                      |
| lon                        | REAL    | Home longitude                     |
| preferred_pay_min          | REAL    | Target floor                       |
| preferred_pay_max          | REAL    | Target ceiling                     |
| max_weekly_commute_miles   | REAL    | Soft preference                    |
| notes                      | TEXT    | Free text                          |

### `jobs` table
| Column                  | Type    | Notes                                      |
|-------------------------|---------|--------------------------------------------|
| job_id                  | TEXT    | PRIMARY KEY (UUID or string)               |
| title                   | TEXT    |                                            |
| company                 | TEXT    |                                            |
| level                   | TEXT    | junior / mid / senior / staff              |
| employment_type         | TEXT    | full_time / part_time / contract / internship |
| remote_policy           | TEXT    | onsite / hybrid / remote                   |
| pay_usd_min             | REAL    |                                            |
| pay_usd_max             | REAL    |                                            |
| location_city           | TEXT    |                                            |
| location_state          | TEXT    |                                            |
| office_lat              | REAL    | Nullable                                   |
| office_lon              | REAL    | Nullable                                   |
| days_in_office_per_week | REAL    | 0 for remote                               |
| one_way_commute_miles   | REAL    | 0 if remote                                |
| weekly_commute_miles    | REAL    | Derived: one_way × 2 × days                |
| skills                  | TEXT    | Comma-separated or simple string           |
| manual_match            | INTEGER | 1–10 user estimate                         |
| job_score               | REAL    | Final weighted score (0–100)               |
| score_pay               | REAL    | Component                                  |
| score_commute           | REAL    | Component                                  |
| score_match             | REAL    | Component                                  |
| source_url              | TEXT    | Nullable                                   |
| date_posted             | TEXT    | ISO YYYY-MM-DD                             |
| date_added              | TEXT    | ISO YYYY-MM-DD                             |
| raw_data_path           | TEXT    | Nullable                                   |
| notes                   | TEXT    |                                            |

### `applications` table
| Column            | Type    | Notes                                      |
|-------------------|---------|--------------------------------------------|
| application_id    | TEXT    | PRIMARY KEY                                |
| job_id            | TEXT    | FK → jobs.job_id                           |
| title             | TEXT    | Denormalized (optional)                    |
| company           | TEXT    | Denormalized (optional)                    |
| job_score         | REAL    | Snapshot                                   |
| status            | TEXT    | saved / applied / interviewing / offered / accepted / rejected_employer / rejected_self / withdrawn |
| applied_date      | TEXT    | ISO                                        |
| last_contact_date | TEXT    | ISO                                        |
| next_follow_up    | TEXT    | ISO – drives reminders                     |
| interview_stage   | INTEGER | 0 = none, 1 = first, …                     |
| offer_date        | TEXT    | ISO                                        |
| offer_pay         | REAL    |                                            |
| rejected_by       | TEXT    | employer / self / empty                    |
| notes             | TEXT    |                                            |
| archived          | INTEGER | 0/1                                        |

**Foreign key:** `FOREIGN KEY (job_id) REFERENCES jobs(job_id)`

---

## 4. Scoring Logic (V1)

**Weekly commute**
```text
weekly_commute_miles = one_way_commute_miles * 2 * days_in_office_per_week
```
- remote → days = 0 → weekly miles = 0

**Component scores** (normalize to ~0–1 or 0–100)  
1. **Pay** – higher when offer is inside/above preferred range  
2. **Commute** – higher when weekly miles are low  
3. **Match** – `manual_match / 10`

**Final score example**
```text
job_score = 100 * (
    0.40 * pay_norm +
    0.25 * commute_norm +
    0.35 * match_norm
)
```
Store all three component scores + final `job_score` on the job row.

---

## 5. Project Structure

```text
star_hound_tracker/
├── job_tracker.ipynb          ← this notebook (prototype & test)
├── job_tracker.py             ← main CLI entry point (later)
├── python/
│   ├── db.py                  ← connect, init_db, PRAGMA
│   ├── users.py
│   ├── scoring.py
│   ├── jobs.py
│   ├── applications.py
│   ├── reminders.py
│   ├── viz.py
│   └── ... (V2 modules)
├── data/                      ← gitignored
│   └── jobs.db
├── plots/                     ← generated charts
├── resumes/                   ← V2
└── reports/                   ← V2
```

---

## 6. Recommended Implementation Order

| # | Module            | Goal                                              | Status |
|---|-------------------|---------------------------------------------------|--------|
| 1 | `db.py`           | `init_db()` + `get_connection()`                  | Done   |
| 2 | `users.py`        | Create / view / update user profile               |        |
| 3 | `scoring.py`      | Pure functions for commute + weighted score       |        |
| 4 | `jobs.py`         | Add / list / view jobs (with scoring)             |        |
| 5 | `applications.py` | Track status, dates, notes, follow-ups            |        |
| 6 | `reminders.py`    | Follow-up checklist                               |        |
| 7 | `viz.py`          | First 2–3 charts                                  |        |
| 8 | `job_tracker.py`  | Simple CLI menu that calls the modules            |        |

---

## 7. Design Principles

- SQLite is the **source of truth**; pandas is only for analysis & plotting
- Prefer a single `status` field + dates over many boolean flags
- **Never delete** rejected applications — set status and optionally archive
- Store score **components** so weights can be tuned later
- Use **parameterized SQL** (`?` placeholders) for every user input
- Enable foreign keys on every connection
- Keep V1 fully useful without scraping or NLP
- Everything personal stays local and gitignored

---

## 8. How to Use This Notebook

```python
# Typical starting cells
from python.db import init_db, get_connection
init_db()                          # safe to run multiple times

# Later, as modules are written:
# from python.users import create_user, get_user
# from python.scoring import calculate_scores
# from python.jobs import add_job, list_jobs
```

**Workflow tip**  
1. Write pure functions first (hard-coded test data)  
2. Verify they write/read correctly from SQLite  
3. Only then add `input()` prompts  
4. Move working code into the proper `python/` module

---

## 9. Quick Reset

To start completely fresh during development:

```python
import os
from pathlib import Path

db_path = Path("data/jobs.db")
if db_path.exists():
    db_path.unlink()
    print("Database deleted.")

from python.db import init_db
init_db()
```

---

**Current focus:** Finish `users.py` → then `scoring.py` → then `jobs.py`
```

You can copy the entire block above and paste it into a single Markdown cell at the top of your notebook. It will serve as a living reference while you build.